In this lab, you will be building a Plotly Dash application for users to perform interactive visual analytics on SpaceX launch data in real-time.

This dashboard application contains input components such as a dropdown list and a range slider to interact with a pie chart and a scatter point chart. You will be guided to build this dashboard application via the following tasks:

- TASK 1: Add a Launch Site Drop-down Input Component
- TASK 2: Add a callback function to render success-pie-chart based on selected site dropdown
- TASK 3: Add a Range Slider to Select Payload
- TASK 4: Add a callback function to render the success-payload-scatter-chart scatter plot

In [1]:
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

In [2]:
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv")

spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


In [3]:
min_payload = spacex_df['Payload Mass (kg)'].min()
max_payload = spacex_df['Payload Mass (kg)'].max()

In [4]:
# Create Dash application

app = dash.Dash(__name__)



# ---------------------------------------------------------
# TASK 1: Add a Launch Site Drop-down Input Component
# ---------------------------------------------------------

app.layout = html.Div([

    html.H1(
        "SpaceX Launch Dashboard",
        style={'textAlign': 'center'}
    ),

    html.Br(),

    html.Label("Select Launch Site:"),

    dcc.Dropdown(
        id='site-dropdown',

        # Dropdown options include All Sites
        # and individual launch sites

        options=[
            {'label': 'All Sites', 'value': 'ALL'}
        ] +
        [
            {'label': site, 'value': site}
            for site in spacex_df['Launch Site'].unique()
        ],

        # Default value

        value='ALL',

        # Placeholder text

        placeholder="Select a Launch Site here",

        # Enable searching

        searchable=True
    ),


    html.Br(),


    # ---------------------------------------------------------
    # TASK 3: Add a Range Slider to Select Payload
    # ---------------------------------------------------------

    html.Label("Select Payload Range (Kg):"),


    dcc.RangeSlider(

        id='payload-slider',

        # Minimum payload

        min=0,

        # Maximum payload

        max=10000,

        # Slider interval

        step=1000,


        marks={
            0: '0',
            2500: '2500',
            5000: '5000',
            7500: '7500',
            10000: '10000'
        },


        # Default selected range

        value=[min_payload, max_payload]

    ),


    html.Br(),


    # Graph for pie chart

    dcc.Graph(
        id='success-pie-chart'
    ),


    # Graph for scatter plot

    dcc.Graph(
        id='success-payload-scatter-chart'
    )

])





# ---------------------------------------------------------
# TASK 2: Add callback function to render success-pie-chart
# based on selected site dropdown
# ---------------------------------------------------------


@app.callback(

    Output(
        component_id='success-pie-chart',
        component_property='figure'
    ),

    Input(
        component_id='site-dropdown',
        component_property='value'
    )

)


def get_pie_chart(entered_site):


    # If ALL sites are selected

    if entered_site == 'ALL':


        fig = px.pie(

            spacex_df,

            values='class',

            names='Launch Site',

            title='Total Successful Launches by Site'

        )


    # If a specific site is selected

    else:


        # Filter dataframe for selected site

        filtered_df = spacex_df[
            spacex_df['Launch Site'] == entered_site
        ]


        fig = px.pie(

            filtered_df,

            values='class',

            names='class',

            title=f'Success Rate for {entered_site}'

        )


    return fig





# ---------------------------------------------------------
# TASK 4: Add callback function to render
# success-payload-scatter-chart scatter plot
# ---------------------------------------------------------

@app.callback(
    Output(
        component_id='success-payload-scatter-chart',
        component_property='figure'
    ),
    [
        Input(
            component_id='site-dropdown',
            component_property='value'
        ),
        Input(
            component_id='payload-slider',
            component_property='value'
        )
    ]
)

def update_scatter(selected_site, payload_range):

    # Get payload range from slider
    min_payload = payload_range[0]
    max_payload = payload_range[1]


    # Filter dataframe based on payload range
    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= min_payload) &
        (spacex_df['Payload Mass (kg)'] <= max_payload)
    ]


    # Filter by selected launch site
    if selected_site != 'ALL':

        filtered_df = filtered_df[
            filtered_df['Launch Site'] == selected_site
        ]


    # Create scatter plot

    fig = px.scatter(
        filtered_df,
        x='Payload Mass (kg)',
        y='class',
        color='Booster Version Category',
        title='Payload Mass vs Launch Outcome'
    )


    return fig





# ---------------------------------------------------------
# Run the application
# ---------------------------------------------------------

if __name__ == '__main__':

    app.run()